In [ ]:
# 4 Particular Dataset Optimization
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

print(" Starting XGBoost Hyperparameter Tuning for ALL 4 Datasets...\n")


datasets = {
    "LFI": "/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv",
    "XSS": "/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv",
    "OSC": "/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv",
    "SQL": "/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv"
}

tuning_results = []

for dataset_name, file_path in datasets.items():
    print(f"\n{'='*80}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*80}")

    # Load data
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].fillna(0)
    y = df.iloc[:, -1].values

    print(f"Shape: {df.shape} | Class Distribution: {pd.Series(y).value_counts().to_dict()}")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Hyper_perameter_Tuning_RandomizedSearchCV
    xgb = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

    param_dist = {
        'n_estimators': [200, 300, 400, 500],
        'max_depth': [4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.15],
        'subsample': [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'gamma': [0, 0.1, 0.2],
        'min_child_weight': [1, 3, 5]
    }

    print(f"Starting tuning for {dataset_name}... ")

    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_dist,
        n_iter=100,
        cv=10,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    random_search.fit(X_train_scaled, y_train)

    # Best model
    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test_scaled)


    # Training predictions
    y_train_pred = best_model.predict(X_train_scaled)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='macro')
    test_f1 = f1_score(y_test, y_pred, average='macro')

    acc_gap = train_acc - test_acc
    f1_gap = train_f1 - test_f1

    print(f"\n Overfitting Analysis for {dataset_name}:")
    print(f"   Train Accuracy : {train_acc:.5f} | Test Accuracy : {test_acc:.5f} | Gap: {acc_gap:.5f}")
    print(f"   Train F1-Score : {train_f1:.5f} | Test F1-Score : {test_f1:.5f} | Gap: {f1_gap:.5f}")

    if acc_gap > 0.08:
        print("   HIGH OVERFITTING detected (Train-Test gap > 0.08)")
    elif acc_gap > 0.04:
        print("   Moderate Overfitting detected")
    elif acc_gap < -0.04:
        print("   Possible Underfitting (Test better than Train)")
    else:
        print("    Good Generalization")



    acc = test_acc
    f1 = test_f1

    print(f" {dataset_name} Best Params: {random_search.best_params_}")
    print(f"   Accuracy : {acc:.5f} | F1-Score : {f1:.5f}")

    # Save model and scaler
    joblib.dump(best_model, f'best_xgboost_{dataset_name.lower()}.pkl')
    joblib.dump(scaler, f'scaler_{dataset_name.lower()}.pkl')

    tuning_results.append({
        'Dataset': dataset_name,
        'Accuracy': acc,
        'F1_Score': f1,
        'Best_CV_Score': random_search.best_score_,
        'Train_Test_Acc_Gap': acc_gap,
        'Train_Test_F1_Gap': f1_gap,
        'Overfitting_Risk': 'High' if acc_gap > 0.08 else 'Medium' if acc_gap > 0.04 else 'Low'
    })


summary_df = pd.DataFrame(tuning_results)
summary_df = summary_df.sort_values(by='F1_Score', ascending=False)

print("\n" + "="*90)
print(" FINAL HYPERPARAMETER TUNING SUMMARY")
print("="*90)
print(summary_df.round(5))

summary_df.to_csv('xgboost_tuning_summary_all_datasets.csv', index=False)
print("\n Summary saved to 'xgboost_tuning_summary_all_datasets.csv'")

print("\n All models and scalers have been saved!")

 Starting XGBoost Hyperparameter Tuning for ALL 4 Datasets...


Processing: LFI
Shape: (31177, 8) | Class Distribution: {1: 22716, 0: 8461}
Starting tuning for LFI... 
Fitting 10 folds for each of 100 candidates, totalling 1000 fits

 Overfitting Analysis for LFI:
   Train Accuracy : 0.93898 | Test Accuracy : 0.93842 | Gap: 0.00055
   Train F1-Score : 0.92545 | Test F1-Score : 0.92498 | Gap: 0.00047
    Good Generalization
 LFI Best Params: {'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.9}
   Accuracy : 0.93842 | F1-Score : 0.92498

Processing: XSS
Shape: (30461, 7) | Class Distribution: {1: 19770, 0: 10691}
Starting tuning for XSS... 
Fitting 10 folds for each of 100 candidates, totalling 1000 fits

 Overfitting Analysis for XSS:
   Train Accuracy : 0.99696 | Test Accuracy : 0.99803 | Gap: -0.00107
   Train F1-Score : 0.99667 | Test F1-Score : 0.99784 | Gap: -0.00117
    Good Generalization
 XSS 

In [ ]:
# XGBoost + RandomizedSearchCV on Merged Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

data_path = "/content/drive/MyDrive/WAF_THESIS/DATASETS/merged_all_attacks.csv"
df = pd.read_csv(data_path)
print(f"Dataset Loaded: {df.shape}")
print("Attack Type Distribution:\n", df['attack_type'].value_counts())
print("Class Distribution:\n", df['class'].value_counts())

target_col = 'class'
# Features (exclude target and metadata)
feature_cols = [col for col in df.columns if col not in [target_col, 'attack_type']]
X = df[feature_cols]
y = df[target_col]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training Shape: {X_train.shape} | Test Shape: {X_test.shape}")

print("\n Starting Hyperparameter Tuning with RandomizedSearchCV...")

# Parameter grid
param_dist = {
    'n_estimators': [200, 300, 400, 500, 600],
    'max_depth': [4, 6, 8, 10, 12],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [0, 0.1, 1, 10]
}

# Base model
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1
)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100, # Number of random combinations to try
    scoring='f1_macro',
    cv=10, # 10-fold cross validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# Best parameters
print("\n" + "="*60)
print("BEST PARAMETERS FOUND")
print("="*60)
print(random_search.best_params_)
print(f"Best Cross-Validation F1-Score: {random_search.best_score_:.4f}")

best_params = random_search.best_params_

final_model = XGBClassifier(
    **best_params,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1
)

print("\nTraining Final Model with Best Parameters...")
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

# OVERFITTING / UNDERFITTING ANALYSIS
y_train_pred = final_model.predict(X_train)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_pred)
train_f1 = f1_score(y_train, y_train_pred, average='macro')
test_f1 = f1_score(y_test, y_pred, average='macro')

acc_gap = train_acc - test_acc
f1_gap = train_f1 - test_f1

print("\n" + "="*60)
print("OVERFITTING / UNDERFITTING ANALYSIS")
print("="*60)
print(f"Train Accuracy : {train_acc:.4f} | Test Accuracy : {test_acc:.4f} | Gap: {acc_gap:.4f}")
print(f"Train F1-Score : {train_f1:.4f} | Test F1-Score : {test_f1:.4f} | Gap: {f1_gap:.4f}")

if acc_gap > 0.10:
    print(" HIGH OVERFITTING DETECTED (Gap > 0.10)")
elif acc_gap > 0.05:
    print(" Moderate Overfitting Detected")
elif acc_gap < -0.04:
    print(" Possible Underfitting (Test performing better than Train)")
else:
    print(" Good Generalization - Low Overfitting Risk")

print("="*60)

#Final_Evaluation
print("\n" + "="*70)
print("FINAL MODEL PERFORMANCE (XGBoost Tuned)")
print("="*70)
print(f"Accuracy : {accuracy_score(y_test, y_pred)*100:.4f}%")
print(f"F1-Score : {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

import joblib
joblib.dump(final_model, "/content/drive/MyDrive/WAF_THESIS/MODELS/best_xgboost_tuned.pkl")
print("\n Best XGBoost model saved successfully!")

Dataset Loaded: (94210, 21)
Attack Type Distribution:
 attack_type
LFI    31177
XSS    30461
OSC    19095
SQL    13477
Name: count, dtype: int64
Class Distribution:
 class
1    56452
0    37758
Name: count, dtype: int64
Training Shape: (75368, 19) | Test Shape: (18842, 19)

 Starting Hyperparameter Tuning with RandomizedSearchCV...
Fitting 10 folds for each of 100 candidates, totalling 1000 fits

BEST PARAMETERS FOUND
{'subsample': 0.9, 'reg_lambda': 0, 'reg_alpha': 0.5, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0.3, 'colsample_bytree': 0.7}
Best Cross-Validation F1-Score: 0.9603

Training Final Model with Best Parameters...

OVERFITTING / UNDERFITTING ANALYSIS
Train Accuracy : 0.9619 | Test Accuracy : 0.9608 | Gap: 0.0011
Train F1-Score : 0.9603 | Test F1-Score : 0.9591 | Gap: 0.0012
 Good Generalization - Low Overfitting Risk

FINAL MODEL PERFORMANCE (XGBoost Tuned)
Accuracy : 96.0832%
F1-Score : 0.9591

Classification Report:
      

In [ ]:


import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore")


model_path = "/content/drive/MyDrive/WAF_THESIS/MODELS/best_xgboost_tuned.pkl"

print("Loading XGBoost model...")
model = joblib.load(model_path)
print(" Model loaded successfully!\n")


# Change this path to your new dataset
new_data_path = "/content/TEST_set.csv"   # or .csv

if new_data_path.endswith('.xlsx'):
    new_df = pd.read_excel(new_data_path)
else:
    new_df = pd.read_csv(new_data_path)

print(f"New Dataset Shape: {new_df.shape}")
print("New Dataset Columns:", new_df.columns.tolist())

# Get features the model was trained on
trained_features = model.feature_names_in_   # XGBoost stores this

print(f"Model was trained on {len(trained_features)} features")

# Align new data
aligned_df = new_df.copy()

# Add missing columns with 0
for col in trained_features:
    if col not in aligned_df.columns:
        aligned_df[col] = 0

# Remove extra columns not known by model
extra_cols = [col for col in aligned_df.columns if col not in trained_features
              and col not in ['class', 'attack_type']]
if extra_cols:
    aligned_df = aligned_df.drop(columns=extra_cols)


aligned_df = aligned_df[trained_features]

print(f"Aligned Data Shape: {aligned_df.shape}")
print(" Columns aligned with training data")


y_pred = model.predict(aligned_df)

print("\n Predictions completed!")

if 'class' in new_df.columns:
    y_true = new_df['class'].values

    print("\n" + "="*70)
    print("MODEL PERFORMANCE ON NEW DATASET")
    print("="*70)
    print(f"Accuracy  : {accuracy_score(y_true, y_pred)*100:.4f}%")
    print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1-Score  : {f1_score(y_true, y_pred, average='macro'):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
else:
    print("No 'class' column found. Only predictions were made.")

aligned_df['predicted_class'] = y_pred
aligned_df.to_csv("/content/predictions_new_dataset.csv", index=False)

print("\n Predictions saved to: predictions_new_dataset.csv")

Loading XGBoost model...
✅ Model loaded successfully!

New Dataset Shape: (20, 7)
New Dataset Columns: ['sign1', 'sign2', 'sign3', 'sign4', 'sign5', 'sign6', 'badwords']
Model was trained on 19 features
Aligned Data Shape: (20, 19)
✅ Columns aligned with training data

✅ Predictions completed!
No 'class' column found. Only predictions were made.

💾 Predictions saved to: predictions_new_dataset.csv
